# Classical time series analysis
This notebook tracks my efforts to find a competitive classical time series model that can serve as a benchmark against which I'd like to compare my quantum reservoir computing model.

In [1]:
import os.path

from src.data.loading.configs import MODEL_DIR_CLASSICAL
# ---- load data ----
from src.data.loading.geosphere import Geosphere

data = Geosphere().load_data_into_memory()

historically_relevant_data = data[-1000:]

Loading data from file


In [2]:
# ---- select data ----

import pandas as pd

selected_columns = [
    'cglo'
    , 'chim'
    , 'dd'
    , 'ddx'
    , 'ff'
#    , 'ffam_flag'
    , 'ffx'
    , 'p'
#    , 'pred'
    , 'rf'
    , 'rr'
    , 'rrm'
    , 'sh'
    , 'so'
    , 'tb10'
    , 'tb20'
    , 'tb50'
    , 'tl'
    , 'tlmax'
    , 'tlmin'
    , 'ts'
    , 'tsmax'
    , 'tsmin'
    , 'zeitx'
    , 'timestamps'
#    , 'stationId'
]

usable_data = historically_relevant_data.dropna(axis="columns", how="all")

final_data_columns = usable_data.columns.intersection(selected_columns)

final_data = historically_relevant_data[final_data_columns].reset_index(drop=True)
final_data.set_index(pd.to_datetime(final_data['timestamps'], format="ISO8601"))
final_data.index.freq = '10min'

# output
print("Selected columns: {}".format(selected_columns))
print("Usable columns: {}".format(usable_data.columns))
print("Realised columns: {}".format(final_data_columns))

Selected columns: ['cglo', 'chim', 'dd', 'ddx', 'ff', 'ffx', 'p', 'rf', 'rr', 'rrm', 'sh', 'so', 'tb10', 'tb20', 'tb50', 'tl', 'tlmax', 'tlmin', 'ts', 'tsmax', 'tsmin', 'zeitx', 'timestamps']
Usable columns: Index(['cglo', 'dd', 'ddx', 'ff', 'ffam_flag', 'ffx', 'p', 'pred', 'rf', 'rr',
       'rrm', 'so', 'tl', 'tlmax', 'tlmin', 'ts', 'tsmax', 'tsmin', 'zeitx',
       'timestamps', 'stationId'],
      dtype='object')
Realised columns: Index(['cglo', 'dd', 'ddx', 'ff', 'ffx', 'p', 'rf', 'rr', 'rrm', 'so', 'tl',
       'tlmax', 'tlmin', 'ts', 'tsmax', 'tsmin', 'zeitx', 'timestamps'],
      dtype='object')


In [ ]:
import pickle
# ---- model selection ----

# parameters of the time-series:
# - one-dimensional
# - value 'tl'

# tried but failed due to conflicting dependencies:
# Seshadri, Ram (2020). GitHub - AutoViML/Auto_TS: enables you to build and deploy multiple time
# series models using ML and statistical techniques with a single line of code.
# Source code: https://github.com/AutoViML/Auto_TS

# instead used:
import pmdarima as pm

# reload/select best model
file = MODEL_DIR_CLASSICAL / 'arima_model.pkl'

if os.path.exists(file):    # load
    classical_model = pickle.load(open(file, "rb"))
else:   # select best model according to the data and fits model to data
    classical_model = pm.auto_arima(final_data['tl'],
                        seasonal=True,
                        m=144, # 144 Intervalle à 10min = 24 Stunden Saison
                        error_action='ignore',
                        suppress_warnings=True,
                        stepwise=True)
    # dump arima model to avoid recalculation
    with open(file, 'wb') as pkl:
        pickle.dump(classical_model, pkl)

print("The type of the model is: {}".format(type(classical_model)))
print("Its parameters are: {}".format(classical_model))

In [1]:
# ---- model prediction ----
#MM: to be continued
forecast, conf_int = classical_model.predict(n_periods=50, return_conf_int=True)

print("Forecast: {}".format(forecast))
print("Conf int: {}".format(conf_int))


NameError: name 'classical_model' is not defined